In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
import uuid

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, END, START, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

c:\Users\Ashutosh Pandey\Desktop\Machine Learning\LangGraph-In-Depth\LangGraph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
store = InMemoryStore()

In [ ]:
# for creating memory also we need llm
Extractor_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [ ]:
#memory deciding factor pydantic
class MemoryDecision(BaseModel):
    should_store: bool = Field(description="wheather to store the memory")
    memories: list[str] = Field(default_factory=list,description="list of memories to store")

In [ ]:
memory_extractor = Extractor_llm.with_structured_output(MemoryDecision)

: 

In [ ]:
# creating the chat node function
def chatnode(state: MessagesState, config: RunnableConfig, store: BaseStore):
    # taking the thread id:
    user_id = config["configurable"]["user_id"]
    
    # creating the namespace:
    namespace = ("user", user_id, "details")
    
    #getting the last message from the response
    last_message = state["messages"][-1].content
    
    